In [2]:
import os
import shutil
import pandas as pd
import numpy as np

This section reads 2 VtV files and prepares them to be merged. method == 0 reads files that end with _single.txt, these are the single speaker transcriptions produced from videos where there is only 1 speaker. method != 0 reads files that end with _single_X.txt and _single_Y.txt, these are single speaker transcriptions produced by splitting the speakers from videos with 2 speakers.

In [4]:
def read_transcription_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line for line in f if not line.startswith('#') and line.strip() != '']


    from io import StringIO
    data_str = ''.join(lines)
    df = pd.read_csv(StringIO(data_str), sep='|', header=None, names=['Start Time', 'End Time', 'Speaker', 'Transcription'])

    df = df.map(lambda x: x.strip() if isinstance(x, str) else x)

    return df


In [5]:
method = 0
if method == 0: # Merge seperate transcriptions
    directory = "Test/full/VtV/"
    file_start = "GS1-VtV1"
    
    dfA = read_transcription_file(directory + file_start + "A_full.txt")
    dfB = read_transcription_file(directory + file_start + "B_full.txt")
else: # Merge pre split transcriptions
    directory = "Transcripts/Study 2 (Fall2023)/VTV/"
    file_start = "VtV17"
    
    dfA = pd.read_csv(directory + "single/"+ file_start + "A_single_X.txt", sep="|")
    dfB = pd.read_csv(directory + "single/"+ file_start + "B_single_Y.txt", sep="|")

To merge, we select 2 timestamps in both transcriptions that have been manually determined to be identical in actual time. We then calculate the difference and subtract the difference from the first transcriptions file, which synchronizes the timestamps for both files.

In [48]:
dfA_time = 7.500
dfB_time = 122.24000000000001
difference = dfA_time - dfB_time

In [49]:
dfA["Start Time"] = dfA["Start Time"] -difference
dfA["End Time"]= dfA["End Time"] -difference
dfA["Speaker"] = "Speaker 1"
dfB["Speaker"] = "Speaker 2"

We can then merge both files and then sort by the start time of each line of speech. This produces a merged transcription that we can now save to a csv file.

In [50]:
df_new = pd.concat([dfA, dfB])
df_new = df_new.sort_values(by=["Start Time"])
df_new.to_csv(directory + "dyad/"+ file_start + "_dyad.txt", index=False,sep="|")

# Merge videos to each other

In [1]:
from pydub import AudioSegment
import os

def combine_dual_recordings(
    file_a,
    file_b,
    output_path="combined.wav",
    align_offset_ms=0,
    stereo=True,
    target_sample_rate=16000
):
    """
    Combines two mono recordings (e.g., from two speakers) into a single audio file.
    
    Parameters:
        file_a (str): Path to speaker A's audio file
        file_b (str): Path to speaker B's audio file
        output_path (str): Output WAV path for the combined audio
        align_offset_ms (int): Offset in milliseconds to delay speaker B (can be negative)
        stereo (bool): If True, saves as stereo (speaker A = left, B = right). If False, mixes into mono
        target_sample_rate (int): Output sample rate (Hz), default is 16000
    """
    print(f"🔄 Loading audio files: {file_a} and {file_b}")
    a = AudioSegment.from_file(file_a).set_frame_rate(target_sample_rate).set_channels(1)
    b = AudioSegment.from_file(file_b).set_frame_rate(target_sample_rate).set_channels(1)

    # Apply the offset
    if align_offset_ms > 0:
        print(f"⏱️ Delaying speaker B by {align_offset_ms} ms")
        b = AudioSegment.silent(duration=align_offset_ms) + b
    elif align_offset_ms < 0:
        print(f"⏱️ Delaying speaker A by {-align_offset_ms} ms")
        a = AudioSegment.silent(duration=-align_offset_ms) + a

    # Equalize lengths by padding
    max_len = max(len(a), len(b))
    a += AudioSegment.silent(duration=max_len - len(a))
    b += AudioSegment.silent(duration=max_len - len(b))

    # Combine (stereo option)
    if stereo:
        print("🔊 Combining as stereo (A = left, B = right)")
        combined = AudioSegment.from_mono_audiosegments(a, b)
    else:
        print("🔊 Combining as mono overlay")
        combined = a.overlay(b)

    # Export
    combined.export(output_path, format="wav")
    print(f"✅ Combined audio saved to: {output_path}")

In [10]:
study_a = "./Data/DECEPTION/ATA/NATA41 (SUSPECTED).m4a"
study_b = "./Data/DECEPTION/ATA/NATA41-R (UNPINNED).mp4"
combine_dual_recordings(
    file_a=study_a,
    file_b=study_b,
    output_path="combined_3_s.wav",
    align_offset_ms=0,  # Set this manually if there's a delay
    stereo=True
)

🔄 Loading audio files: ./Data/DECEPTION/ATA/NATA41 (SUSPECTED).m4a and ./Data/DECEPTION/ATA/NATA41-R (UNPINNED).mp4
🔊 Combining as stereo (A = left, B = right)
✅ Combined audio saved to: combined_3_s.wav


# Q/A merge technique

In [1]:
!pip install sentence-transformers

In [21]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
from datetime import datetime, timedelta
import re

model = SentenceTransformer('all-MiniLM-L6-v2')

def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def parse_transcript(file_path, speaker_name):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line for line in f if not line.strip().startswith('#')]
    from io import StringIO
    df = pd.read_csv(StringIO(''.join(lines)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    utterances = []
    for _, row in df.iterrows():
        utterances.append({
            'speaker': speaker_name,
            'start': time_to_seconds(row['Start Time']),
            'end': time_to_seconds(row['End Time']),
            'text': row['Transcription']
        })
    return utterances

def extract_questions(utterances):
    return [u for u in utterances if '?' in u['text'] or re.match(
        r'(?i)\b(what|how|when|where|why|who|do|did|does|are|is|can|could|would|should)\b',
        u['text'])]

def find_best_qa_alignment(questions, responses, max_delay=90):
    q_texts = [q['text'] for q in questions]
    r_texts = [r['text'] for r in responses]

    q_embeddings = model.encode(q_texts, convert_to_tensor=True)
    r_embeddings = model.encode(r_texts, convert_to_tensor=True)

    best_score = -1
    best_pair = None
    for i, q in enumerate(questions):
        sims = util.cos_sim(q_embeddings[i], r_embeddings)[0]
        j = sims.argmax().item()
        print(q_texts[i], '|', r_texts[j], '|', best_score)
        
        if sims[j] > best_score:
            best_score = sims[j]
            if abs(responses[j]['start'] - q['start']) < max_delay:
                best_pair = (q, responses[j])

    if best_pair:
        offset = best_pair[1]['start'] - best_pair[0]['start']
        return offset, best_pair, best_score.item()
    return None, None, None

def apply_offset(utterances, offset):
    return [
        {**u, 'start': u['start'] - offset, 'end': u['end'] - offset}
        for u in utterances
    ]

def merge_transcripts(utterances_a, utterances_b):
    all_utts = utterances_a + utterances_b
    all_utts.sort(key=lambda x: x['start'])
    return pd.DataFrame([{
        'Start Time': seconds_to_time(u['start']),
        'End Time': seconds_to_time(u['end']),
        'Speaker': u['speaker'],
        'Text': u['text']
    } for u in all_utts])


In [22]:
from pathlib import Path
data_path = Path('./Test/GSGS#1/GS1-VtV1/full')
ali_path, bailey_path = data_path / 'GS1-VtV1A_full.txt', data_path / 'GS1-VtV1B_full.txt'

ali_utts = parse_transcript(ali_path, "Ali")
bailey_utts = parse_transcript(bailey_path, "Bailey")

offset, pair, score = find_best_qa_alignment(
        extract_questions(bailey_utts), ali_utts
    )
print(pair)
print(f"Matched QA:\nQ: {pair[0]['text']}\nA: {pair[1]['text']}\nScore: {score:.2f}, Offset: {offset:.2f}s")

aligned_ali = ali
aligned_bailey = apply_offset(bailey, offset)

merged = merge_transcripts(aligned_ali, aligned_bailey)
merged.to_csv("merged_output.csv", index=False)

/tmp/ipykernel_727/1180558789.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_727/1180558789.py:21: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


What's happening? It's happening. | She's up. | -1
your giving in which is Bailey. Can she hear me? Yeah, can you hear her? | She's up. | tensor(0.3566, device='cuda:0')
Hi, how are you? | I'm good, how are you? | tensor(0.3566, device='cuda:0')
What are you studying? | What are you taking? | tensor(0.7741, device='cuda:0')
cool what year are you? | How old are you? | tensor(0.7741, device='cuda:0')
Are you living in residential? | Oh, that's really cool. How was um, residence? I heard like um, the, like the rent. It's like really expensive. It's like around like 7,000. | tensor(0.7741, device='cuda:0')
And then it's so weird. So what classes are you taking this year? | Oh, that's cool. I didn't get to dress up this year, cause I don't know, I wasn't thinking about it, cause there's so many like assignments and I had a class, so I was tired. | tensor(0.7741, device='cuda:0')
When I'm learning in law, that's a really good question. I don't know yet. I haven't been paid attention very mu

TypeError: 'NoneType' object is not subscriptable

In [18]:
extract_questions(bailey_utts)

[{'speaker': 'Bailey',
  'start': 22.379,
  'end': 23.849,
  'text': "What's happening? It's happening."},
 {'speaker': 'Bailey',
  'start': 90.939,
  'end': 96.649,
  'text': 'your giving in which is Bailey. Can she hear me? Yeah, can you hear her?'},
 {'speaker': 'Bailey',
  'start': 110.62,
  'end': 111.53,
  'text': 'Hi, how are you?'},
 {'speaker': 'Bailey',
  'start': 132.139,
  'end': 133.289,
  'text': 'What are you studying?'},
 {'speaker': 'Bailey',
  'start': 136.46,
  'end': 137.37,
  'text': 'cool what year are you?'},
 {'speaker': 'Bailey',
  'start': 166.62,
  'end': 168.49,
  'text': 'Are you living in residential?'},
 {'speaker': 'Bailey',
  'start': 232.539,
  'end': 236.09,
  'text': "And then it's so weird. So what classes are you taking this year?"},
 {'speaker': 'Bailey',
  'start': 277.98,
  'end': 284.25,
  'text': "When I'm learning in law, that's a really good question. I don't know yet. I haven't been paid attention very much."},
 {'speaker': 'Bailey',
  'sta

In [1]:
import pandas as pd
from datetime import datetime, timedelta

def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def parse_transcript(file_path, speaker_name):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line for line in f if not line.strip().startswith('#')]

    from io import StringIO
    df = pd.read_csv(StringIO(''.join(lines)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    utterances = []
    for _, row in df.iterrows():
        utterances.append({
            'speaker': speaker_name,
            'text': row['Transcription'],
            'start': time_to_seconds(row['Start Time']),
            'end': time_to_seconds(row['End Time']),
            'original_start': row['Start Time'],
            'original_end': row['End Time']
        })
    return utterances

def apply_offset(utterances, offset):
    return [
        {
            **u,
            'start': u['start'] - offset,
            'end': u['end'] - offset
        }
        for u in utterances
    ]

def merge_transcripts(utterances_a, utterances_b):
    all_utts = utterances_a + utterances_b
    all_utts.sort(key=lambda x: x['start'])

    return pd.DataFrame([{
        'Start Time': u['original_start'],
        'End Time': u['original_end'],
        'Speaker': u['speaker'],
        'Text': u['text']
    } for u in all_utts])

In [16]:
from pathlib import Path
numb = 17
data_path = Path(f'./Test/GSGS#1/GS1-VtV{numb}/')

ali_path, bailey_path = data_path / f'single/GS1-VtV{numb}A_single.txt', data_path / f'single/GS1-VtV{numb}B_single.txt'

ali_utts = parse_transcript(ali_path, "Ali")
bailey_utts = parse_transcript(bailey_path, "Bailey")

t_question = "00:01:21.289"   # from ali
t_response = "00:01:20.540"   # from bailey

# Compute and apply offset
offset = time_to_seconds(t_response) - time_to_seconds(t_question)
print(f"Manual offset: {offset:.2f} seconds")

t_response_aligned = apply_offset(bailey_utts, offset)

# Merge
merged_df = merge_transcripts(ali_utts, t_response_aligned)
merged_df.to_csv(f"{data_path}/dyad/merged_conversation_manual.txt", sep='|', index=False)

Manual offset: -0.75 seconds


/tmp/ipykernel_69132/3748664534.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_69132/3748664534.py:18: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


1. VtV2 -> only one speaker
2. VtV3 -> Not quite accurate. Hard to tell exactly what is the conversation (especially in VtV3B)
3. VtV7 -> Only one Speaker
4. VtV8 -> Only one speaker
5. VtV9 -> Not quite accurate.
6. 

Postprocessing

In [13]:
import pandas as pd
from datetime import datetime, timedelta

def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def conditional_role_mapping(df, mapping_dict):
    df = df.copy()
    # First pass: change Speaker Role based on Speaker values
    if 'Speaker' in mapping_dict:
        for speaker_val, new_role in mapping_dict['Speaker'].items():
            df.loc[df['Speaker'] == speaker_val, 'Speaker Role'] = new_role

    # Second pass: update Speaker Role based on existing Role values
    if 'Speaker Role' in mapping_dict:
        for old_role, new_role in mapping_dict['Speaker Role'].items():
            df.loc[df['Speaker Role'] == old_role, 'Speaker Role'] = new_role

    return df

def transcript_mapper_conditional(file_path, output_path, single_output, role, mapping_dict):
    # Read lines and metadata
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    metadata = [line for line in lines if line.strip().startswith('#')]
    content = [line for line in lines if not line.strip().startswith('#')]

    from io import StringIO
    df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    df_mapped = conditional_role_mapping(df, mapping_dict)

    with open(output_path, 'w', encoding='utf-8') as f:
        for line in metadata:
            f.write(line)
        df_mapped.to_csv(f, sep='|', index=False, header=False)


    # ========== SINGLE VERSION ==========
    df_single = df_mapped[df_mapped['Speaker Role'].str.lower() == role.lower()].copy()
    df_single['Speaker'] = role
    # df_single['Speaker Role'] = ''  # Clear role for single version

    # Save single speaker output
    df_single.to_csv(single_output, sep='|', index=False, header=False)

    print(f"Saved full transcript to: {output_path}")
    print(f"Saved single-speaker transcript to: {single_output}")

In [15]:
from pathlib import Path
numb = 17
Test/GSGS#1/GS1-FtF(SUSPECTED 2023 FEB 02 4 32 PM)/full/GS1-FtF(SUSPECTED 2023 FEB 02 4 32 PM)-033_full.txt
data_path = Path(f'./Test/GSGS#1/')
bailey_single_path, bailey_full_path = data_path / f'single/GS1-VtV{numb}B_single.txt', data_path / f'full/GS1-VtV{numb}B_full.txt'
mapping_dict = {
    'Speaker':{
        'Speaker 2': 'Bailey',
        'Speaker 1': 'Others'
    }
}

transcript_mapper_conditional(bailey_full_path, bailey_full_path, bailey_single_path, 'Bailey', mapping_dict)

Saved full transcript to: Test/GSGS#1/GS1-VtV17/full/GS1-VtV17B_full.txt
Saved single-speaker transcript to: Test/GSGS#1/GS1-VtV17/single/GS1-VtV17B_single.txt


/tmp/ipykernel_69132/2194027996.py:36: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [9]:
import pandas as pd
from datetime import datetime, timedelta

def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def preprocess_transcript_dual(file_path, full_output, single_output):
    # Read lines, skipping metadata
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line for line in f if not line.strip().startswith('#')]

    from io import StringIO
    df = pd.read_csv(StringIO(''.join(lines)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # ========== FULL VERSION ==========
    df_full = df.copy()

    # Rename Speaker Role from Bailey to Ali
    df_full.loc[df_full['Speaker Role'].str.lower() == 'bailey', 'Speaker Role'] = 'Ali'
    # Rename speaker_2 to Bailey
    df_full.loc[df_full['Speaker'].str.lower() == 'speaker 2', 'Speaker Role'] = 'Bailey'


    # Save full output
    df_full.to_csv(full_output, sep='|', index=False, header=False)

    # ========== SINGLE VERSION ==========
    df_single = df_full[df_full['Speaker Role'].str.lower() == 'bailey'].copy()
    df_single['Speaker'] = 'Bailey'
    # df_single['Speaker Role'] = ''  # Clear role for single version

    # Save single speaker output
    df_single.to_csv(single_output, sep='|', index=False, header=False)

    print(f"Saved full transcript to: {full_output}")
    print(f"Saved single-speaker transcript to: {single_output}")

# Run it
from pathlib import Path
numb = 15
data_path = Path(f'./Test/GSGS#1/GS1-VtV{numb}/')
bailey_single_path, bailey_full_path = data_path / f'single/GS1-VtV{numb}A_single.txt', data_path / f'full/GS1-VtV{numb}B_full.txt'
bailey_single_path2, bailey_full_path2 = data_path / f'single/GS1-VtV{numb}B_single_2.txt', data_path / f'full/GS1-VtV{numb}B_full_2.txt'

preprocess_transcript_dual(bailey_full_path, bailey_full_path2, bailey_single_path2)

Saved full transcript to: Test/GSGS#1/GS1-VtV15/full/GS1-VtV15B_full_2.txt
Saved single-speaker transcript to: Test/GSGS#1/GS1-VtV15/single/GS1-VtV15A_single_2.txt


/tmp/ipykernel_69132/895031762.py:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


# Switch between speakers

In [5]:
def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def parse_transcript(file_path, speaker_name):
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = [line for line in f if not line.strip().startswith('#')]
    from io import StringIO
    df = pd.read_csv(StringIO(''.join(lines)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
    utterances = []
    for _, row in df.iterrows():
        utterances.append({
            'speaker': speaker_name,
            'start': time_to_seconds(row['Start Time']),
            'end': time_to_seconds(row['End Time']),
            'text': row['Transcription']
        })
    return utterances

def interleave_from_anchor_strict(q_list, a_list, q_start, a_start):
    """
    Build an interleaved transcript starting from a known Q/A match,
    adding consecutive utterances from one speaker until a gap appears,
    then switching to the other speaker.
    
    Parameters:
    - q_list: list of utterances from the question speaker (e.g., Ali)
    - a_list: list of utterances from the answer speaker (e.g., Bailey)
    - q_start: start time (in seconds) of the known question
    - a_start: start time (in seconds) of the known answer
    
    Returns:
    - merged_df: DataFrame containing the interleaved transcript
    """
    timeline = []

    # Find anchor indices
    q_index = next(i for i, u in enumerate(q_list) if abs(u['start'] - q_start) < 0.01)
    a_index = next(i for i, u in enumerate(a_list) if abs(u['start'] - a_start) < 0.01)
    print(q_index, a_index)
    current_speaker = 'Q'  # start with the speaker who asked the question
    last_end = None

    while q_index < len(q_list) or a_index < len(a_list):
        print(f"index Q: {q_index}, index A: {a_index}, current speaker: {current_speaker}")
        if current_speaker == 'Q':
            while q_index < len(q_list):
                utt = q_list[q_index]
                if last_end is None or utt['start'] >= last_end:
                    timeline.append(utt)
                    last_end = utt['end']
                    q_index += 1
                    # check if next q starts after a gap with last_end
                    if q_index < len(q_list) and q_list[q_index]['start'] > last_end:
                        break
                else:
                    q_index += 1
            current_speaker = 'A'
            last_end = None

        elif current_speaker == 'A':
            while a_index < len(a_list):
                utt = a_list[a_index]
                if last_end is None or utt['start'] >= last_end:
                    timeline.append(utt)
                    last_end = utt['end']
                    a_index += 1
                    # check if next a starts after a gap with last_end
                    if a_index < len(a_list) and a_list[a_index]['start'] > last_end:
                        break
                else:
                    a_index += 1
            current_speaker = 'Q'
            last_end = None

    return pd.DataFrame([{
        'Start Time': seconds_to_time(u['start']),
        'End Time': seconds_to_time(u['end']),
        'Speaker': u['speaker'],
        'Text': u['text']
    } for u in timeline])

In [53]:
from pathlib import Path
data_path = Path('./Test/GSGS#1/GS1-VtV1/full')
ali_path, bailey_path = data_path / 'GS1-VtV1A_full.txt', data_path / 'GS1-VtV1B_full.txt'

ali_utts = parse_transcript(ali_path, "Ali")
bailey_utts = parse_transcript(bailey_path, "Bailey")

t_question = "00:01:50.620"   # from Bailey
t_response = "00:00:07.580"   # from Ali
q_start = time_to_seconds(t_question)
a_start = time_to_seconds(t_response)

merged_df = interleave_from_anchor_strict(bailey_utts, ali_utts, q_start, a_start)
# merged_df.head(100)
merged_df.to_csv('merged_conversation_manual.txt', index=False, sep='|')

8 1
index Q: 8, index A: 1, current speaker: Q
index Q: 9, index A: 1, current speaker: A
index Q: 9, index A: 2, current speaker: Q
index Q: 10, index A: 2, current speaker: A
index Q: 10, index A: 3, current speaker: Q
index Q: 11, index A: 3, current speaker: A
index Q: 11, index A: 4, current speaker: Q
index Q: 12, index A: 4, current speaker: A
index Q: 12, index A: 5, current speaker: Q
index Q: 13, index A: 5, current speaker: A
index Q: 13, index A: 6, current speaker: Q
index Q: 14, index A: 6, current speaker: A
index Q: 14, index A: 7, current speaker: Q
index Q: 15, index A: 7, current speaker: A
index Q: 15, index A: 8, current speaker: Q
index Q: 16, index A: 8, current speaker: A
index Q: 16, index A: 9, current speaker: Q
index Q: 17, index A: 9, current speaker: A
index Q: 17, index A: 10, current speaker: Q
index Q: 18, index A: 10, current speaker: A
index Q: 18, index A: 11, current speaker: Q
index Q: 19, index A: 11, current speaker: A
index Q: 19, index A: 12, c

/tmp/ipykernel_727/940282232.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_727/940282232.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


In [32]:
def interleave_from_anchor_strict(q_list, a_list, q_start, a_start):
    """
    Build an interleaved transcript starting from a known Q/A match,
    allowing same-speaker continuation if their next utterance is sooner
    than the other speaker’s next.
    """
    timeline = []

    q_index = next(i for i, u in enumerate(q_list) if abs(u['start'] - q_start) < 0.01)
    a_index = next(i for i, u in enumerate(a_list) if abs(u['start'] - a_start) < 0.01)
    print(q_index, a_index)
    current_speaker = 'Q'
    q_last_end, a_last_end = q_list[q_index]['end'], a_list[a_index]['end']
    timeline.append(q_list[q_index]); timeline.append(a_list[a_index])
    q_index += 1
    a_index += 1
    while q_index < len(q_list) or a_index < len(a_list):
        print(q_index, a_index, current_speaker, q_last_end, a_last_end)
        if current_speaker == 'Q':
            while q_index < len(q_list):
                print('\t continue in Q')
                utt = q_list[q_index]
                if q_last_end is None or utt['start'] >= q_last_end:
                    timeline.append(utt)
                    q_last_end = utt['end']
                    q_index += 1

                    # Check who speaks next
                    next_q_gap = (q_list[q_index]['start'] - q_last_end) if q_index < len(q_list) else float('inf')
                    next_a_gap = (a_list[a_index]['start'] - a_last_end) if a_index < len(a_list) else float('inf')

                    if next_a_gap < next_q_gap:
                        break
                else:
                    q_index += 1
            current_speaker = 'A'

        elif current_speaker == 'A':
            while a_index < len(a_list):
                utt = a_list[a_index]
                if a_last_end is None or utt['start'] >= a_last_end:
                    timeline.append(utt)
                    a_last_end = utt['end']
                    a_index += 1

                    # Check who speaks next
                    next_q_gap = (q_list[q_index]['start'] - q_last_end) if q_index < len(q_list) else float('inf')
                    next_a_gap = (a_list[a_index]['start'] - a_last_end) if a_index < len(a_list) else float('inf')

                    if next_a_gap > next_q_gap:
                        break
                else:
                    a_index += 1
            current_speaker = 'Q'

    return pd.DataFrame([{
        'Start Time': seconds_to_time(u['start']),
        'End Time': seconds_to_time(u['end']),
        'Speaker': u['speaker'],
        'Text': u['text']
    } for u in timeline])

In [33]:
from pathlib import Path
data_path = Path('./Test/GSGS#1/GS1-VtV1/full')
ali_path, bailey_path = data_path / 'GS1-VtV1A_full.txt', data_path / 'GS1-VtV1B_full.txt'

ali_utts = parse_transcript(ali_path, "Ali")
bailey_utts = parse_transcript(bailey_path, "Bailey")

t_question = "00:01:50.620"   # from Bailey
t_response = "00:00:07.580"   # from Ali
q_start = time_to_seconds(t_question)
a_start = time_to_seconds(t_response)

merged_df = interleave_from_anchor_strict(bailey_utts, ali_utts, q_start, a_start)
# merged_df.head(100)
merged_df.to_csv('merged_conversation_manual.txt', index=False, sep='|')

8 1
9 2 Q 111.53 8.65
	 continue in Q
10 2 A 113.609 8.65
10 3 Q 113.609 11.849
	 continue in Q
11 3 A 121.049 11.849
11 22 Q 121.049 153.21
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
28 22 A 236.09 153.21
28 73 Q 236.09 506.0
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q
	 continue in Q


/tmp/ipykernel_722/940282232.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
/tmp/ipykernel_722/940282232.py:14: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)


# Postprocessing

In [8]:
import pandas as pd
from datetime import datetime, timedelta
from io import StringIO

def time_to_seconds(time_str):
    t = datetime.strptime(time_str.strip(), "%H:%M:%S.%f")
    return t.hour * 3600 + t.minute * 60 + t.second + t.microsecond / 1e6

def seconds_to_time(seconds):
    return str(timedelta(seconds=seconds))

def compute_metadata(base_metadata, df, num_speakers=None):
    """Update metadata for duration, number of speakers, and number of segments."""
    meta = []
    for line in base_metadata:
        if line.lower().startswith("# duration:"):
            # compute duration from df
            start_time = time_to_seconds(df.iloc[0]['Start Time'])
            end_time = time_to_seconds(df.iloc[-1]['End Time'])
            duration = seconds_to_time(end_time - start_time)
            meta.append(f"# Duration: {duration}\n")
        elif line.lower().startswith("# number of speakers:") and num_speakers is not None:
            meta.append(f"# Number of Speakers: {num_speakers}\n")
        elif line.lower().startswith("# number of segments:"):
            meta.append(f"# Number of Segments: {len(df)}\n")
        else:
            meta.append(line)
    return meta

def conditional_role_mapping(df, mapping_dict):
    """Apply mapping rules to update Speaker Role based on conditions."""
    df = df.copy()

    # Apply Speaker → Speaker Role mapping
    if 'Speaker' in mapping_dict:
        for speaker_val, new_role in mapping_dict['Speaker'].items():
            df.loc[df['Speaker'] == speaker_val, 'Speaker Role'] = new_role

    # Apply Speaker Role remapping
    if 'Speaker Role' in mapping_dict:
        for old_role, new_role in mapping_dict['Speaker Role'].items():
            df.loc[df['Speaker Role'] == old_role, 'Speaker Role'] = new_role

    return df

def save_with_metadata(df, path, metadata, num_speakers=None):
    """Save a dataframe to file with updated metadata."""
    updated_metadata = compute_metadata(metadata, df, num_speakers=num_speakers)
    with open(path, 'w', encoding='utf-8') as f:
        for line in updated_metadata:
            f.write(line)
        df.to_csv(f, sep='|', index=False, header=False)

def transcript_splitter(file_path, mapped_file, bailey_file, ali_file, combined_file, mapping_dict):
    # --- Read input file ---
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    metadata = [line for line in lines if line.strip().startswith('#')]
    content = [line for line in lines if not line.strip().startswith('#')]

    df = pd.read_csv(StringIO(''.join(content)), sep='|', header=None, engine='python')
    df.columns = ['Start Time', 'End Time', 'Speaker', 'Speaker Role', 'Transcription']
    df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)

    # --- Step 1: Apply mapping ---
    df_mapped = conditional_role_mapping(df, mapping_dict)
    # df_mapped = df

    # --- Step 2: Save mapped full transcript (keep original metadata) ---
    save_with_metadata(df_mapped, mapped_file, metadata)

    # --- Step 3: Separate Bailey and Ali ---
    df_bailey = df_mapped[df_mapped['Speaker Role'].str.lower() == 'bailey'].copy()
    df_ali = df_mapped[df_mapped['Speaker Role'].str.lower() == 'ali'].copy()
    # df_bailey = df_mapped[df_mapped['Speaker'].str.lower() == 'speaker 1'].copy()
    # df_ali = df_mapped[df_mapped['Speaker'].str.lower() == 'speaker 2'].copy()

    df_bailey['Speaker'] = 'Bailey'
    df_ali['Speaker'] = 'Ali'

    save_with_metadata(df_bailey, bailey_file, metadata, num_speakers=1)
    save_with_metadata(df_ali, ali_file, metadata, num_speakers=1)

    # --- Step 4: Create combined transcript ---
    df_combined = pd.concat([df_bailey, df_ali]).sort_values(by=['Start Time'])
    save_with_metadata(df_combined, combined_file, metadata, num_speakers=2)

    print(f"✅ Saved mapped full transcript → {mapped_file}")
    print(f"✅ Saved Bailey transcript → {bailey_file}")
    print(f"✅ Saved Ali transcript → {ali_file}")
    print(f"✅ Saved Combined transcript → {combined_file}")

In [18]:
mapping_dict = {
    'Speaker': {
        # 'Speaker 1': 'Paul',   # If Speaker == speaker_2 → Speaker Role = Bailey
        # 'Speaker 2': 'Jeff',
        # 'speaker_2': 'RA'
    },
    # 'Speaker Role': {
    #     'Bailey': 'Ali'        # If Role == Bailey → Speaker Role = Ali
    # }
}

from pathlib import Path
# numb = 17
name = f'GS1-FtF30A'
version = f'-016'
# version = f''
# /home/gnamiro/GSGS/transcription/NemoWhisperTranscriptionPipeline/Test/GSGS#1/GS1-FtF6A/full/GS1-FtF6A-034_full.txt
data_path = Path(f'./Test/GSGS#1/{name}/full')
file_name = data_path / f'{name}{version}_full.txt'
transcript_splitter(
    file_name,
    data_path / f'{name}{version}_full.txt',
    data_path / f'{name}{version}_single_B.txt',
    data_path / f'{name}{version}_single_A.txt',
    data_path / f'{name}{version}_dyad.txt',
    mapping_dict
)

✅ Saved mapped full transcript → Test/GSGS#1/GS1-FtF30A/full/GS1-FtF30A-016_full.txt
✅ Saved Bailey transcript → Test/GSGS#1/GS1-FtF30A/full/GS1-FtF30A-016_single_B.txt
✅ Saved Ali transcript → Test/GSGS#1/GS1-FtF30A/full/GS1-FtF30A-016_single_A.txt
✅ Saved Combined transcript → Test/GSGS#1/GS1-FtF30A/full/GS1-FtF30A-016_dyad.txt


/tmp/ipykernel_336437/2091377164.py:64: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: x.strip() if isinstance(x, str) else x)
